# Nettoyage de la base LTM — Atelier BI, Sprint 1
**Projet :** Power BI LTM (amortisseurs) · **Environnement :** Google Colab

Ce notebook prend la base brute `LTM_base_.xlsx` (7 feuilles : `dim_produit`, `dim_vehicule`, `pont_compatibilite`, `fact_prix`, `fact_ventes`, `dim_client`, `dim_date`) et produit `LTM_base_adaptee.xlsx`, avec un journal de toutes les modifications.

**Comment l'utiliser**
1. Exécute les cellules dans l'ordre (Exécution > Tout exécuter).
2. À la première cellule, importe `LTM_base_.xlsx` quand la fenêtre s'ouvre.
3. Le fichier final se télécharge automatiquement à la dernière cellule.

> La base est **générée (simulée)** : les valeurs ont donc pu être corrigées directement. Ce point doit être déclaré dans le rapport.

## 1. Imports et chargement du fichier
Importe les bibliothèques, demande le fichier brut et charge les 7 feuilles.

In [ ]:
import re
import numpy as np
import pandas as pd

try:
    from google.colab import files          # disponible seulement dans Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    print("Importe le fichier LTM_base_.xlsx :")
    televerse = files.upload()
    SRC = list(televerse.keys())[0]
else:                                        # exécution en local
    SRC = "LTM_base_.xlsx"

OUT = "LTM_base_adaptee.xlsx"

T = pd.read_excel(SRC, sheet_name=None)
p  = T["dim_produit"].copy()
v  = T["dim_vehicule"].copy()
pc = T["pont_compatibilite"].copy()
fp = T["fact_prix"].copy()
fv = T["fact_ventes"].copy()
c  = T["dim_client"].copy()
d  = T["dim_date"].copy()

journal = []
def noter(table, colonne, modification, nb, justification):
    journal.append([table, colonne, modification, int(nb), justification])

Importe le fichier LTM_base_.xlsx :


Saving LTM_base_adaptee (1).xlsx to LTM_base_adaptee (1).xlsx


## 2. Diagnostic avant nettoyage
Compte les anomalies de la base d'origine. Ces chiffres servent au diagnostic qualité (L9).

In [ ]:
def diagnostic(p, v, pc, fp, fv):
    ltm = p.fabricant.eq("LTM")
    non_actifs = set(p[ltm & (p.statut != "Actif")].id_produit)
    m = fv.merge(fp[["id_produit", "date_prix"]], on="id_produit", how="left")
    return {
        "nulls dim_produit (cote)":            int(p.cote.isna().sum()),
        "nulls dim_produit (tendance)":        int(p.tendance.isna().sum()),
        "nulls dim_produit (rang_demande)":    int(p.rang_demande.isna().sum()),
        "vehicules segment 'Non renseigné'":   int((v.segment == "Non renseigné").sum()),
        "vehicules sans produit compatible":   int((~v.id_vehicule.isin(pc.id_vehicule)).sum()),
        "ventes sur produits non actifs":      int(fv.id_produit.isin(non_actifs).sum()),
        "ventes avant la date du prix":        int((m.date_facture < m.date_prix).sum()),
        "CA total (TND)":                      round(float(fv.montant_ht_tnd.sum()), 2),
    }

avant = diagnostic(p, v, pc, fp, fv)
print("DIAGNOSTIC AVANT NETTOYAGE")
for k, val in avant.items():
    print(f"  {k:40s} {val}")

noter("(toutes)", "(toutes)", "Base reconnue comme générée (simulée)", 0,
      "Les valeurs ont pu être corrigées directement. À déclarer dans l'inventaire des sources (L6) et dans le rapport")

DIAGNOSTIC AVANT NETTOYAGE
  nulls dim_produit (cote)                 621
  nulls dim_produit (tendance)             0
  nulls dim_produit (rang_demande)         0
  vehicules segment 'Non renseigné'        0
  vehicules sans produit compatible        0
  ventes sur produits non actifs           0
  ventes avant la date du prix             0
  CA total (TND)                           1052955.04


## 3. `dim_produit` : colonne `cote`
Les nulls correspondent aux essieux « Avant » et « Arrière » sans côté : absence normale, remplacée par « N/A ».

In [ ]:
n = p.cote.isna().sum()
p["cote"] = p["cote"].fillna("N/A")
noter("dim_produit", "cote", 'Nulls remplacés par "N/A"', n,
      "Essieux Avant / Arrière sans côté : absence normale")

## 4. `dim_produit` : tendance
Les 5 valeurs manquantes sont déduites de l'évolution des quantités vendues entre 2025 et 2026 (Stable si aucune vente).

In [ ]:
ltm = p.fabricant.eq("LTM")
q25 = fv[fv.date_facture < "2026-01-01"].groupby("id_produit").quantite.sum()
q26 = fv[fv.date_facture >= "2026-01-01"].groupby("id_produit").quantite.sum()
delta = (q26.reindex(p.id_produit).fillna(0).values
         - q25.reindex(p.id_produit).fillna(0).values)
a_des_ventes = p.id_produit.isin(fv.id_produit).values
manquants = p.tendance.isna().values
remplissage = np.where(a_des_ventes, np.sign(delta), 0)   # 0 = Stable si aucune vente
p.loc[manquants, "tendance"] = remplissage[manquants]
p["tendance"] = p.tendance.astype(int)
p["tendance_libelle"] = p.tendance.map({1: "Hausse", -1: "Baisse", 0: "Stable"})
noter("dim_produit", "tendance",
      "5 nulls remplis : signe de l'évolution des quantités vendues 2026 vs 2025 (0 si aucune vente)",
      manquants.sum(),
      "PRD00027, 48, 65 : ventes arrêtées en 2026 donc -1 ; PRD00033, 57 : aucune vente donc 0 (Stable)")
noter("dim_produit", "tendance_libelle", "Colonne ajoutée (Hausse / Baisse / Stable)", len(p),
      "Libellé lisible pour Power BI")

## 5. `fact_ventes` : ventes sur produits non actifs
Les ventes de produits Obsolète ou En développement sont réaffectées à un produit Actif de même gamme et de même type. Le prix et le montant sont recalculés.

In [ ]:
# -> réaffectées à un produit Actif de même gamme et même type (prix et montant recalculés)
non_actifs = set(p[ltm & (p.statut != "Actif")].id_produit)
actifs = p[ltm & (p.statut == "Actif")]
masque = fv.id_produit.isin(non_actifs)

compteur, nouveaux = {}, []
for pid in fv.loc[masque, "id_produit"]:
    r = p[p.id_produit == pid].iloc[0]
    cand = actifs[(actifs.gamme == r.gamme) & (actifs.type_amortisseur == r.type_amortisseur)]
    if cand.empty:
        cand = actifs[actifs.gamme == r.gamme]
    ids = sorted(cand.id_produit)
    cle = (r.gamme, r.type_amortisseur)
    k = compteur.get(cle, 0)
    compteur[cle] = k + 1
    nouveaux.append(ids[k % len(ids)])              # répartition à tour de rôle
fv.loc[masque, "id_produit"] = nouveaux

prix = fp.set_index("id_produit").prix_tnd
fv["prix_unitaire_tnd"] = fv.id_produit.map(prix)
fv["montant_ht_tnd"] = (fv.quantite * fv.prix_unitaire_tnd * (1 - fv.remise_pct / 100)).round(2)
noter("fact_ventes", "id_produit, prix_unitaire_tnd, montant_ht_tnd",
      "Ventes de produits Obsolète / En développement réaffectées à un produit Actif de même gamme et même type ; prix et montant recalculés",
      masque.sum(),
      "Un produit obsolète ou en développement ne peut pas être vendu. Quantités, dates, clients et remises inchangés ; CA total modifié par le changement de prix")

## 6. `fact_prix` : date du prix
La date du prix est ramenée à la date de la première vente, pour qu'aucune vente ne précède son prix.

In [ ]:
premiere = fv.groupby("id_produit").date_facture.min()
date_vente = fp.id_produit.map(premiere)
a_corriger = date_vente < fp.date_prix
fp.loc[a_corriger, "date_prix"] = date_vente[a_corriger]
noter("fact_prix", "date_prix", "date_prix ramenée à la date de la première vente du produit",
      a_corriger.sum(), "Le prix catalogue doit être valide au moment de la première vente")

## 7. `dim_produit` : rang de demande
Pour LTM, le rang est le classement des quantités vendues (1 = plus vendu), règle retrouvée dans la base d'origine. Les rangs KYB ne changent pas.

In [ ]:
# Règle retrouvée dans la base : rang = classement des quantités vendues (1 = plus vendu)
qt = fv.groupby("id_produit").quantite.sum()
rang = p.loc[ltm, "id_produit"].map(qt).fillna(0).rank(ascending=False, method="min")
p.loc[ltm, "rang_demande"] = rang
p["rang_demande"] = p.rang_demande.astype(int)
noter("dim_produit", "rang_demande",
      "Rang des produits LTM recalculé d'après les quantités vendues (1 = plus vendu), après correction des ventes",
      ltm.sum(),
      "La règle d'origine a été retrouvée (rang = classement des quantités vendues). Elle permet de combler les 2 nulls (PRD00033, 57 : aucune vente) et suit maintenant les ventes corrigées")
noter("dim_produit", "rang_demande", "Non modifié pour KYB", (~ltm).sum(),
      "Rang KYB de 1 à 1120, issu du catalogue : échelle différente de celle de LTM, non comparable entre fabricants")

## 8. `dim_vehicule` : véhicules sans produit
Supprime les véhicules qui n'ont aucun produit compatible.

In [ ]:
orphelins = v[~v.id_vehicule.isin(pc.id_vehicule)]
v = v[v.id_vehicule.isin(pc.id_vehicule)].copy()
noter("dim_vehicule", "(lignes)",
      f"{len(orphelins)} véhicules sans produit compatible supprimés : " + ", ".join(orphelins.id_vehicule),
      len(orphelins), "Aucun lien avec les produits, sans utilité pour l'analyse")

## 9. `dim_vehicule` : segment
Remplace « Non renseigné » par un segment déduit de la marque et du modèle. À relire, en particulier les modèles ambigus (Berlingo, Note, 5008, Octavia).

In [ ]:
REGLES = {
 "Acura": [(".", "SUV")],
 "Alfa Romeo": [("159", "Berline"), ("Brera", "Coupé"), ("Giulia", "Berline"), ("Giulietta", "Compacte"), ("Spider", "Sportive"), ("Stelvio", "SUV")],
 "Aston Martin": [(".", "Citadine")],
 "Audi": [("^A3", "Compacte"), ("^A4", "Berline"), ("^A5", "Coupé"), ("^Q", "SUV")],
 "BMW": [("Compact", "Compacte"), ("Série 1", "Compacte"), ("Série 2", "Coupé"), ("Série 3", "Berline"), ("Série 4", "Coupé"), ("Série 5", "Berline"), ("Z3", "Sportive")],
 "Chevrolet": [(".", "SUV")],
 "Citroën": [("Berlingo", "Monospace"), ("^C1", "Citadine"), ("C25", "VUL"), ("^C3", "Citadine"), ("Cactus", "SUV"), ("^C4", "Compacte"), ("^C5", "Berline"), ("Xantia", "Berline")],
 "Cupra": [(".", "SUV")],
 "DS": [("DS 4", "Compacte"), ("DS 5", "Berline"), ("DS 7", "SUV"), ("DS 9", "Berline")],
 "Dacia": [(".", "Citadine")],
 "Dodge": [("Ram", "Pick-up")],
 "Fiat": [("124", "Sportive"), ("500L", "Monospace"), ("500X", "SUV"), ("^500", "Citadine"), ("Ducato", "VUL"), ("Egea", "Berline"), ("Punto", "Citadine"), ("Linea", "Berline"), ("Panda", "Citadine"), ("Sedici", "SUV"), ("Tipo", "Compacte")],
 "Ford": [("C-Max", "Monospace"), ("Fiesta", "Citadine"), ("Focus", "Compacte"), ("Mazda2", "Citadine"), ("Mazda3", "Compacte"), ("Tourneo", "Monospace"), ("Transit", "VUL")],
 "Honda": [("CR-V", "SUV"), ("City", "Berline"), ("Civic", "Compacte"), ("FR-V", "Monospace"), ("Fit|Jazz", "Citadine"), ("HR-V|Vezel", "SUV")],
 "Hyundai": [("Matrix", "Monospace"), ("Santa Fe|Tucson|Veracruz|ix35", "SUV"), ("Sonata", "Berline"), ("i20", "Citadine"), ("i30", "Compacte"), ("ix20", "Monospace")],
 "Isuzu": [(".", "Pick-up")],
 "Jeep": [(".", "SUV")],
 "Kia": [("Carens", "Monospace"), ("Ceed", "Compacte"), ("Optima", "Berline"), ("Rio|Venga", "Citadine"), ("Sorento|Sportage", "SUV")],
 "Lancia": [(".", "Citadine")],
 "Land Rover": [(".", "SUV")],
 "Lexus": [("GS", "Berline"), ("NX|RX", "SUV"), ("SC", "Coupé")],
 "Mazda": [("CX-5", "SUV"), ("MX-5|Miata|RX-7|RX-8", "Sportive"), ("Mazda6", "Berline")],
 "Mercedes": [("Classe A", "Compacte"), ("Classe [CES]", "Berline"), ("GLK", "SUV"), ("Sprinter", "VUL")],
 "Mitsubishi": [("ASX|Outlander|PHEV", "SUV"), ("Delica|L300", "VUL"), ("Lancer", "Berline")],
 "Nissan": [("Cabstar|NT400|NV400|Primastar", "VUL"), ("Juke|Qashqai", "SUV"), ("Leaf|Micra", "Citadine"), ("Note", "Monospace"), ("Primera|Sentra", "Berline")],
 "Opel": [("Adam|Corsa", "Citadine"), ("Combo|Movano|Vivaro", "VUL"), ("Grandland|Mokka", "SUV")],
 "Peugeot": [("^108|^20[678]", "Citadine"), ("^2008|^3008", "SUV"), ("^30[78]", "Compacte"), ("^40[567]|^508", "Berline"), ("5008", "Monospace"), ("J5|Partner", "VUL")],
 "Renault": [("Captur", "SUV"), ("Clio", "Citadine"), ("Laguna|Latitude|Talisman", "Berline"), ("Master|Trafic", "VUL"), ("Mégane", "Compacte"), ("Scénic", "Monospace")],
 "Seat": [("Leon", "Compacte"), ("Toledo", "Berline")],
 "Skoda": [(".", "Compacte")],
 "Subaru": [("Forester|Outback", "SUV"), ("Justy", "Citadine"), ("Legacy", "Berline")],
 "Suzuki": [("Celerio|Ignis|Swift|Wagon", "Citadine"), ("Vitara|S-Cross", "SUV"), ("Kizashi", "Berline"), ("Swace", "Compacte")],
 "Toyota": [("4Runner|Harrier|Highlander|RAV4", "SUV"), ("Auris|Corolla|Prius", "Compacte"), ("Avensis|Crown", "Berline"), ("Aygo|Vitz|Yaris|iQ", "Citadine"), ("Supra", "Sportive"), ("Verso", "Monospace")],
 "Volkswagen": [("Bora", "Berline"), ("California|Crafter|Transporter", "VUL"), ("Golf", "Compacte"), ("Multivan", "Monospace"), ("Taro", "Pick-up")],
 "Volvo": [("V40", "Compacte"), ("V60|V70", "Berline"), ("XC", "SUV")],
}

def deduire_segment(ligne):
    if ligne.segment != "Non renseigné":
        return ligne.segment                        # segments existants conservés
    for motif, seg in REGLES.get(ligne.marque, []):
        if re.search(motif, ligne.modele):
            return seg
    return "Non renseigné"                          # marque/modèle inconnu : à compléter à la main

nb = (v.segment == "Non renseigné").sum()
v["segment"] = v.apply(deduire_segment, axis=1)
restants = v[v.segment == "Non renseigné"]
if len(restants):
    print("Segments non déduits (à compléter) :")
    print(restants[["id_vehicule", "marque", "modele"]].to_string(index=False))
noter("dim_vehicule", "segment",
      '"Non renseigné" remplacé par un segment déduit de la marque et du modèle', nb - len(restants),
      "Attribution d'après la connaissance générale du marché automobile : à relire. Les segments déjà renseignés sont inchangés")

## 10. `dim_client` : zone
« Maghreb » devient « Maghreb hors Tunisie » pour ne plus chevaucher « Tunisie ».

In [ ]:
m = c.zone.eq("Maghreb")
c.loc[m, "zone"] = "Maghreb hors Tunisie"
noter("dim_client", "zone", '"Maghreb" renommé "Maghreb hors Tunisie"', m.sum(),
      'Évite le chevauchement avec "Tunisie"')

## 11. `fact_ventes` : valeurs extrêmes
Les valeurs extrêmes (règle IQR) sont signalées dans une colonne, pas supprimées : elles restent plausibles en B2B.

In [ ]:
def seuil_iqr(s):
    q1, q3 = s.quantile([.25, .75])
    return q3 + 1.5 * (q3 - q1)

sq, sm = seuil_iqr(fv.quantite), seuil_iqr(fv.montant_ht_tnd)
extreme = (fv.quantite > sq) | (fv.montant_ht_tnd > sm)
fv["alerte_valeur_extreme"] = np.where(extreme, "Oui", "Non")
noter("fact_ventes", "alerte_valeur_extreme", "Colonne ajoutée (Oui / Non, règle IQR 1,5)", extreme.sum(),
      f"Quantité > {sq:.1f} ou montant > {sm:.0f} TND : plausible en B2B, signalé et non supprimé")
noter("fact_prix", "prix_original, devise_originale", "Non modifié", len(fp),
      "Toujours TND, prix_original = prix_tnd : redondant mais pas une erreur. Un seul prix par produit : pas d'historique de prix")
noter("pont_compatibilite / dim_date", "(toutes)", "Non modifié", len(pc) + len(d), "Aucune anomalie")

## 12. Contrôles après nettoyage
Tous les contrôles doivent afficher OK.

In [ ]:
produits_actifs = set(p[p.statut == "Actif"].id_produit)
m = fv.merge(fp[["id_produit", "date_prix", "prix_tnd"]], on="id_produit")
controles = {
    "Aucun null dans aucune table":
        sum(int(df.isna().sum().sum()) for df in [p, v, pc, fp, fv, c, d]) == 0,
    "Plus de ventes sur produits non actifs (LTM)":
        not fv.id_produit.isin(set(p[(p.fabricant == "LTM") & (p.statut != "Actif")].id_produit)).any(),
    "Aucune vente avant la date du prix":
        not (m.date_facture < m.date_prix).any(),
    "Prix de vente = prix catalogue":
        bool((m.prix_unitaire_tnd == m.prix_tnd).all()),
    "montant_ht = quantité x prix x (1 - remise)":
        bool(((fv.quantite * fv.prix_unitaire_tnd * (1 - fv.remise_pct / 100)).round(2) - fv.montant_ht_tnd).abs().max() < 0.011),
    "Clés étrangères valides (ventes -> produit, client, date)":
        bool(fv.id_produit.isin(p.id_produit).all() and fv.id_client.isin(c.id_client).all() and fv.date_facture.isin(d.date).all()),
    "Clés étrangères valides (pont -> produit, véhicule)":
        bool(pc.id_produit.isin(p.id_produit).all() and pc.id_vehicule.isin(v.id_vehicule).all()),
    "Aucun véhicule sans produit":
        bool(v.id_vehicule.isin(pc.id_vehicule).all()),
    "Aucun segment 'Non renseigné'":
        not (v.segment == "Non renseigné").any(),
}
print("\nCONTRÔLES APRÈS NETTOYAGE")
for k, ok in controles.items():
    print(f"  [{'OK' if ok else 'ERREUR'}] {k}")
print(f"\nCA total : avant {avant['CA total (TND)']:,.2f} TND -> après {fv.montant_ht_tnd.sum():,.2f} TND")


CONTRÔLES APRÈS NETTOYAGE
  [OK] Aucun null dans aucune table
  [OK] Plus de ventes sur produits non actifs (LTM)
  [OK] Aucune vente avant la date du prix
  [OK] Prix de vente = prix catalogue
  [OK] montant_ht = quantité x prix x (1 - remise)
  [OK] Clés étrangères valides (ventes -> produit, client, date)
  [OK] Clés étrangères valides (pont -> produit, véhicule)
  [OK] Aucun véhicule sans produit
  [OK] Aucun segment 'Non renseigné'

CA total : avant 1,052,955.04 TND -> après 1,052,955.04 TND


## 13. Export Excel et téléchargement
Écrit les 7 feuilles corrigées et le journal des transformations, puis lance le téléchargement.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

J = pd.DataFrame(journal, columns=["table", "colonne", "modification", "lignes_concernees", "justification"])
feuilles = [("dim_produit", p), ("dim_vehicule", v), ("pont_compatibilite", pc), ("fact_prix", fp),
            ("fact_ventes", fv), ("dim_client", c), ("dim_date", d), ("journal_transformations", J)]
with pd.ExcelWriter(OUT, engine="openpyxl") as w:
    for nom, df in feuilles:
        df.to_excel(w, sheet_name=nom, index=False)

wb = load_workbook(OUT)
fond = PatternFill("solid", start_color="1F4E78")
for ws in wb:
    for cell in ws[1]:
        cell.font = Font(name="Arial", bold=True, color="FFFFFF")
        cell.fill = fond
        cell.alignment = Alignment(vertical="center", wrap_text=True)
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = Font(name="Arial", size=10)
            if hasattr(cell.value, "year"):
                cell.number_format = "DD/MM/YYYY"
    for i, col in enumerate(ws.columns, 1):
        L = max(len(str(x.value)) if x.value is not None else 0 for x in list(col)[:300])
        ws.column_dimensions[get_column_letter(i)].width = min(max(L + 2, 10), 70 if ws.title == "journal_transformations" else 32)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
for row in wb["journal_transformations"].iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
wb.save(OUT)
print(f"\nFichier créé : {OUT}")

if EN_COLAB:
    files.download(OUT)


Fichier créé : LTM_base_adaptee.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>